# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/menna890/-Explaining-Search-Performance-Gaps-Using-Ranking-Signals/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')



In [4]:
import pandas as pd

# Define the path to your parquet file in Google Drive
file_path = '/content/drive/ml_features_march_2026 (1).parquet'

# Load the parquet file into a pandas DataFrame
df = pd.read_parquet(file_path)

# Display the first few rows of the DataFrame
print("DataFrame loaded successfully. Here's the head:")
print(df.head())
print(df.columns)

DataFrame loaded successfully. Here's the head:
            client_hash_id           content_hash_id  avg_position  \
0  client_62f4a7e64f5e0096  content_2e6360ad20fd7107      5.908100   
1  client_62f4a7e64f5e0096  content_ac8663da7484669a      6.419872   
2  client_62f4a7e64f5e0096  content_d49a012dcb924e31      5.177774   
3  client_62f4a7e64f5e0096  content_614baf2af4330bd7      4.685335   
4  client_62f4a7e64f5e0096  content_4a1ca0fa5c177e0c      5.333333   

   total_clicks  total_impressions       ctr  days_with_data  ga4_sessions  \
0           1.0              884.0  0.001131              27           0.0   
1           0.0               28.0  0.000000              13           0.0   
2           0.0              329.0  0.000000              31           0.0   
3           1.0              772.0  0.001295              31           0.0   
4           0.0               12.0  0.000000               8           0.0   

   engaged_sessions  scroll_events  ...  days_since_update  \


## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [8]:
df_dataset = df.copy()

In [9]:
# 1. Typical CTR
typical_ctr = df_dataset['total_clicks'].sum() / df_dataset['total_impressions'].sum()

# 2. Expected clicks
df_dataset['expected_clicks'] = df_dataset['total_impressions'] * typical_ctr

# 3. Underperformance Score
df_dataset['underperformance_score'] = (
    df_dataset['expected_clicks'] - df_dataset['total_clicks']
).clip(lower=0)

# 4. Binary Proxy
df_dataset['is_underperforming'] = (
    df_dataset['total_clicks'] / df_dataset['expected_clicks']
) < 0.5


In [10]:

# This cell is for CODE (numbers, a query, a check).# 1. Define Target and prevent Data Leakage
TARGET = 'underperformance_score'

# 2. Exclude identifiers, label-derived columns, and analytics outcomes (Leakage Prevention)
EXCLUDED_COLS = [
    'client_hash_id', 'content_hash_id', # Context/IDs
    'total_clicks', 'total_impressions', 'ctr', 'expected_clicks', 'is_underperforming', # Label-derived
    'ga4_sessions', 'engaged_sessions', 'scroll_events', 'sessions_organic', 'sessions_ai', 'total_ai_sessions' # Post-prediction outcomes
]

# 3. Select valid Features (Available BEFORE prediction)
NUMERIC_FEATURES = [
    'word_count', 'search_volume', 'backlinks', 'category_count', 'competition', 'cpc',
    'content_age_days', 'days_since_update', 'days_since_optimized', 'ever_optimized',
    'has_keyword_data', 'has_word_count', 'has_search_volume', 'has_backlinks',
    'has_category_count', 'has_competition'
]

CATEGORICAL_FEATURES = ['content_type', 'main_intent', 'provider_used']

# 4. Build the Feature Vector (X) and Target (y)
X_raw = df_dataset[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy()
y = df_dataset[TARGET].copy()

# 5. Handle Missing Values and Categorical Encoding
# Fill remaining NaNs in numeric columns with 0 (since we have has_* flags)
X_raw[NUMERIC_FEATURES] = X_raw[NUMERIC_FEATURES].fillna(0)

# One-Hot Encode categorical columns (drop_first=False to keep all information for tree models)
X = pd.get_dummies(X_raw, columns=CATEGORICAL_FEATURES, dummy_na=True, dtype=int)

print(f"Feature Vector Built Successfully!")
print(f"Number of rows: {X.shape[0]}")
print(f"Number of features: {X.shape[1]}")
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Feature Vector Built Successfully!
Number of rows: 175304
Number of features: 32


In [12]:
# 1. Leakage Check: Prove that no label-derived column sneaked into the features
suspect_columns = [col for col in X.columns if any(leak_term in col for leak_term in ['click', 'impression', 'ctr', 'session'])]

if len(suspect_columns) == 0:
    print(" Leakage Check Passed: No label-derived or sibling columns found in features.")
else:
    print(f" WARNING: Suspect columns found in features: {suspect_columns}")

# 2. Check for missing values after imputation
remaining_missing = X.isnull().sum().sum()
if remaining_missing == 0:
    print(" Missing Values Check Passed: All NaNs handled successfully.")
else:
    print(f" WARNING: Found {remaining_missing} missing values still present in the feature vector.")

# 3. Print final feature categories (Sample)
print(f"Total numeric features: {len(NUMERIC_FEATURES)}")
print(f"Total encoded categorical features: {X.shape[1] - len(NUMERIC_FEATURES)}")

 Leakage Check Passed: No label-derived or sibling columns found in features.
 Missing Values Check Passed: All NaNs handled successfully.
Total numeric features: 16
Total encoded categorical features: 16


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.